這個錯誤 `IndexError: index 2 is out of bounds for axis 0 with size 2` 是一個非常典型的 **資料長度不對稱** 問題。

### 🔍 錯誤原因分析

從截圖中輸入的代碼看，你選了 3 支股票（2330.TW, 2881.TW, 2888.TW），但系統報錯說「大小只有 2」。這通常發生在：

1. **資料缺失**：其中一支股票（很可能是因為日期區間或代號格式）沒有抓到資料，導致 `yf.download` 只回傳了 2 支股票的價格。
2. **迴圈與清單不匹配**：在繪圖函式 `plot_efficient_frontier` 中，它嘗試遍歷你輸入的 3 個 `tickers`，但計算出來的資產統計數據（`individual_assets`）卻只有 2 筆，當程式跑到第 3 筆（Index 2）時就崩潰了。

---

### 🛠️ 修正方案

我們需要加強程式碼的「健壯性」，確保 `tickers` 的清單與實際下載成功的資料保持一致。請針對以下兩處進行修改：

#### 1. 修改 `load_data` 模組 (確保 tickers 與回傳資料同步)

在下載完資料後，應該根據實際抓到的 `data.columns` 來更新資產清單，而不是死守著使用者輸入的初始清單。

```python
# ==================== 資料載入模組 (修正版) ====================
@st.cache_data
def load_data(tickers, start_date, end_date):
    try:
        raw_data = yf.download(tickers, start=start_date, end=end_date, progress=False)
        
        if isinstance(raw_data.columns, pd.MultiIndex):
            data = raw_data['Close']
        else:
            data = raw_data[['Close']]
            # 如果只抓到一個，這裡需要確保 columns 對齊
        
        # --- 新增關鍵修正：移除完全沒抓到資料的標的 ---
        data = data.dropna(axis=1, how='all')
        actual_tickers = data.columns.tolist() # 取得實際有資料的股票名
        
        data = data.ffill().bfill()
        returns = data.pct_change().dropna()
        
        # 年化統計量
        annual_returns = returns.mean() * 252
        annual_cov = returns.cov() * 252
        annual_vol = np.sqrt(np.diag(annual_cov))
        
        return {
            'prices': data,
            'returns': returns,
            'annual_returns': annual_returns,
            'annual_cov': annual_cov,
            'annual_vol': annual_vol,
            'corr_matrix': returns.corr(),
            'actual_tickers': actual_tickers  # 將實際成功的 tickers 傳回
        }
    except Exception as e:
        st.error(f"資料載入失敗: {str(e)}")
        return None

```

#### 2. 修改 `main()` 中的邏輯呼叫

確保後續繪圖使用的 `tickers` 是「實際抓到資料」的標的。

```python
# 在 main() 函式執行最佳化的地方：
if run_optimization:
    data_dict = load_data(tickers, start_date, end_date)
    if data_dict:
        # --- 修正：使用實際抓到資料的 tickers ---
        active_tickers = data_dict['actual_tickers'] 
        
        returns = data_dict['returns']
        annual_returns = data_dict['annual_returns']
        annual_cov = data_dict['annual_cov']
        
        # ... (中間的最佳化邏輯不變) ...

        # 顯示繪圖時傳入 active_tickers
        with col1:
            eff_frontier = generate_efficient_frontier(returns, annual_returns, annual_cov, risk_free_rate=risk_free_rate)
            ind_assets = pd.DataFrame({'return': annual_returns, 'volatility': data_dict['annual_vol']})
            # 這裡把 tickers 改成 active_tickers
            plot_efficient_frontier(eff_frontier, {'return': ret, 'volatility': vol}, ind_assets, active_tickers, risk_free_rate)
        with col2:
            plot_weights_pie(weights, active_tickers) # 這裡也要改

```

---

### 💡 為什麼會這樣？

台股（.TW）有時候會因為特定的國定假日、或者 `yfinance` 伺服器端抓取某個代號失敗（例如 2888.TW 新光金最近可能因為合併或停止交易等因素資料不穩），導致回傳的 DataFrame 寬度不如預期。

**主要建議：**

* 修改後，即便有一支股票沒抓到，程式也會自動過濾掉它並繼續計算剩下的 2 支，不會直接跳紅字報錯。
* 請檢查你的 Python 環境是否能正常連線下載 `2888.TW` 的資料，有時候手動用 `yf.download("2888.TW")` 測一下就能發現問題點。




這是一個非常經典的投資組合最佳化（Portfolio Optimization）現象。從你提供的執行截圖來看，雖然你輸入了 5 支股票，但結果只有 AAPL、GOOGL 和 TSLA 獲得權重，而 MSFT 和 AMZN 的權重幾乎為 0。

這並非程式錯誤，而是均值-方差模型（Mean-Variance Model）在數學求解後的必然結果。以下是三個主要原因：

### 1. 效益（Risk-Return Profile）被其他資產完全覆蓋

在「最大化夏普比率（Max Sharpe Ratio）」的目標下，演算法會尋找「每一單位風險能換取最高超額報酬」的組合。

* 觀察你的**有效前沿圖（Efficient Frontier）**：圖中有兩個點（可能是 MSFT 或 AMZN）位於藍色曲線的下方且偏右。
* **數學邏輯**：如果資產 A 的預期收益比資產 B 高，且波動率（風險）還比資產 B 低，或者與其他資產相關性太高，演算法為了達到「全球最優解」，會直接將效益較差的資產權重設為 0。

### 2. 相關性過高（High Correlation）

MSFT、AAPL、GOOGL、AMZN 都屬於大型科技股（Big Tech），彼此間的相關係數通常極高（可能高達 $0.8$ 以上）。

* **分散投資陷阱**：當兩支股票走勢高度同步時，模型會認為它們是「同一種資產」。此時模型會選出其中表現最好（夏普值最高）的那一支，並把權重全部給它，因為加入另一支高度相關但報酬稍低的資產，並不能有效降低風險，反而會拉低總報酬。

### 3. 角點解（Corner Solution）與約束條件

你的程式碼中設定了 `bounds = tuple((0, 1) for _ in range(n_assets))`：

* 這意味著**禁止放空（Long-only）**。
* 在禁止放空的約束下，最優化問題經常會出現「角點解」，即某些資產的權重被壓縮到邊界值 $0$。如果允許放空，你可能會看到所有資產都有權重，但某些會是負值。

---

### 💡 如何讓權重更「分散」？

如果你希望每支股票都能分配到權重，可以考慮以下調整方式：

* **修改權重約束（最直接）**：
在 `optimize_max_sharpe` 函式中，將 `bounds` 改為 `(0.05, 1)`。這樣每支股票至少會強迫分配到 $5\%$ 的權重。
* **調整風險偏好**：
改選「🛡️ 最小化波動率」，通常會比「最大化夏普比率」更分散，因為模型會為了抵銷波動而強迫納入更多低相關性資產。
* **縮短/改變時間區間**：
你設定的時間從 2020 年開始，這段期間 TSLA 和 NVDA 等個股的表現極度強勢，會導致模型在回測數據中「過度迷戀」這些高增長個股。

**總結：** 你的程式運作完全正確。這結果正是在告訴你：**根據過去幾年的數據，MSFT 和 AMZN 在你的 5 支選股組合中，對於提升「夏普比率」並沒有顯著貢獻。**